# TP3: Relatório de Progresso e Validação de Modelos

## 1. Estratégia de Modelagem
Neste estágio do projeto, estamos focando em **Análise de Sentimento Multi-classe** (Positivo, Negativo, Neutro) aplicada aos comentários de chat do YouTube.

As técnicas sendo testadas incluem:
- **Zero-Shot Learning com LLMs (vLLM)**: Utilizando modelos de linguagem de larga escala para gerar labels automáticos e explicações.
- **Fine-tuning de BERT (BERTimbau)**: Treinamento supervisionado utilizando uma base de dados rotulada manualmente para capturar nuances do futebol brasileiro.
- **Métricas de Engajamento**: Integração de sentimento com momentum da partida (xT).

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score, classification_report, accuracy_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns

# Configurações de visualização
sns.set_theme(style="whitegrid")

## 2. Carregamento e Preparação dos Dados Rotulados
Cada exemplo foi rotulado por dois anotadores diferentes entre um grupo de 5 (Igor, Ivan, Vitor, Alvaro, Dani).

In [2]:
file_path = '../data/processed/consolidated/manual_labeling_bundesliga_LABELED.xlsx'
xl = pd.ExcelFile(file_path)
raters = ['Igor', 'Ivan', 'Vitor', 'Alvaro', 'Dani']

def map_sentiment(s):
    if not isinstance(s, str): return np.nan
    s = s.strip().lower()
    if 'negativo' in s: return 0
    if 'neutro' in s: return 1
    if 'positivo' in s: return 2
    return np.nan

# Carregar todas as abas
dfs = {}
for rater in raters:
    df = xl.parse(rater)
    df['sentimento_num'] = df['sentimento'].apply(map_sentiment)
    dfs[rater] = df[['comment_id', 'sentimento_num']].dropna()

# Cruzar labels para calcular Kappa
pairs = []
master_df = xl.parse('Master')
for idx, row in master_df.iterrows():
    cid = row['comment_id']
    r_list = [r.strip() for r in str(row['pair']).split(',')]
    if len(r_list) == 2:
        r1, r2 = r_list
        try:
            val1 = dfs[r1][dfs[r1]['comment_id'] == cid]['sentimento_num'].values[0]
            val2 = dfs[r2][dfs[r2]['comment_id'] == cid]['sentimento_num'].values[0]
            pairs.append({'comment_id': cid, 'rater1': r1, 'rater2': r2, 'label1': val1, 'label2': val2})
        except:
            continue

pairs_df = pd.DataFrame(pairs)
print(f"Total de pares recuperados: {len(pairs_df)}")

Total de pares recuperados: 999


## 3. Inter-Rater Reliability (Cohen's Kappa)
O Kappa de Cohen mede a concordância entre dois avaliadores, descontando a concordância ao acaso.

In [3]:
kappa = cohen_kappa_score(pairs_df['label1'], pairs_df['label2'])
print(f"Cohen's Kappa Global: {kappa:.3f}")

def interpret_kappa(k):
    if k < 0: return "Sem concordância"
    if k <= 0.20: return "Leve"
    if k <= 0.40: return "Razoável"
    if k <= 0.60: return "Moderada"
    if k <= 0.80: return "Substancial"
    return "Quase Perfeita"

print(f"Interpretação: {interpret_kappa(kappa)}")

Cohen's Kappa Global: 0.472
Interpretação: Moderada


## 4. Criação do Golden Set
O Golden Set será definido pela concordância entre os dois anotadores. Em caso de divergência, marcaremos para revisão ou usaremos um critério de desempate (ex: priorizar labels não-neutros ou média).

In [4]:
# Golden set: Apenas onde houve concordância absoluta para garantir qualidade máxima
golden_df = pairs_df[pairs_df['label1'] == pairs_df['label2']].copy()
golden_df['final_label'] = golden_df['label1']

# Adicionar a mensagem original para avaliação
full_master = xl.parse('Master')
golden_df = golden_df.merge(full_master[['comment_id', 'mensagem']], on='comment_id')

print(f"Tamanho do Golden Set (concordância total): {len(golden_df)}")

Tamanho do Golden Set (concordância total): 652


## 5. Avaliação do Modelo vLLM
Vamos comparar as predições do modelo vLLM com o nosso Golden Set humano.

In [5]:
vllm_df = pd.read_csv('../data/processed/consolidated/gold_standard_vllm.csv')
# Nota: O vLLM exportado já tem comment_id sequencial ou mensagem correspondente.
# Vamos cruzar por mensagem para garantir o match correto.
vllm_df['mensagem_clean'] = vllm_df['mensagem'].str.strip()
golden_df['mensagem_clean'] = golden_df['mensagem'].str.strip()

eval_df = golden_df.merge(vllm_df[['mensagem_clean', 'sentiment_vllm']], on='mensagem_clean')

print(f"Exemplos para avaliação: {len(eval_df)}")

y_true = eval_df['final_label']
y_pred = eval_df['sentiment_vllm']

print("\n--- Métricas Iniciais (vLLM vs Humano) ---")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.3f}")
print(f"F1-Score (Weighted): {f1_score(y_true, y_pred, average='weighted'):.3f}")
print("\nRelatório de Classificação:")
print(classification_report(y_true, y_pred, target_names=['Negativo', 'Neutro', 'Positivo']))

Exemplos para avaliação: 716

--- Métricas Iniciais (vLLM vs Humano) ---
Accuracy: 0.690
F1-Score (Weighted): 0.674

Relatório de Classificação:
              precision    recall  f1-score   support

    Negativo       0.61      0.96      0.74       267
      Neutro       0.81      0.60      0.69       314
    Positivo       0.82      0.36      0.50       135

    accuracy                           0.69       716
   macro avg       0.74      0.64      0.64       716
weighted avg       0.73      0.69      0.67       716



## 6. Insights Preliminares
- **Dificuldade em Sarcasmo**: Notamos que o vLLM às vezes interpreta gírias de aposta como negativas quando são celebrações.
- **Concordância Humana**: O Kappa obtido indica o quão subjetiva é a tarefa, sugerindo a necessidade de diretrizes de anotação mais claras.
- **Performance**: Resultados iniciais mostram que o modelo base é promissor, mas necessita de ajuste fino para o domínio de futebol.

## 7. Cronograma e Próximos Passos

| Data | Atividade | Objetivo |
| :--- | :--- | :--- |
| 05/Jun | Refinamento do Golden Set | Resolver divergências manuais nos ~10% restantes |
| 10/Jun | Treinamento BERTimbau | Superar baseline do vLLM com modelo especializado |
| 15/Jun | Tuning de Hiperparâmetros | Otimizar Learning Rate e Batch Size |
| 20/Jun | Integração Final | Correlacionar sentimento com xT e Match Momentum |
| 25/Jun | Entrega Final | Relatório consolidado e Dashboard |

**Métricas escolhidas**: F1-Score (Weighted) é a principal, pois as classes são desbalanceadas (muito neutro).